**Prédiction des rendements de fin de session du marché action américain**

Ce notebook implémente un modèle de machine learning pour prédire la direction du rendement des deux dernières heures (entre 14h00 et 16h00) de cotation d'une action.

L'objectif est de prédire la variable "reod " en prenant les valeurs :

- -1 : baisse significative
- 0  : variation faible
- 1  : hausse significative

Le modèle utilise :
- Feature engineering sur les rendements intraday (53 intervalles de 5 minutes)
- LightGBM
- Random Forest
- Stacking des probabilités

Le dataset d'entraînement contient environ 843 000 observations et 53 intradays par observation.

Le score obtenu dépasse le benchmark fourni (41.74%).

------------
Instructions d'utilisation

1. Placer les fichiers suivants dans le même dossier que le notebook :

- x_train.csv
- y_train.csv
- x_test.csv


2. Installer les dépendances : pip install pandas numpy scikit-learn lightgbm.

3. Exécuter les cellules dans l'ordre.

4. Le fichier de soumission sera généré automatiquement : soumission.csv

5. Le fichier peut ensuite être soumis sur la plateforme du challenge.

-----------------


In [1]:
# ======================================================
# IMPORTS
# ======================================================

# Manipulation de données
import pandas as pd
import numpy as np

# Gestion mémoire (utile pour gros datasets)
import gc

# Modèles de machine learning
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# Validation croisée
from sklearn.model_selection import StratifiedKFold

# Métrique d'évaluation
from sklearn.metrics import accuracy_score

In [2]:
# ======================================================
# CHARGEMENT DES DONNÉES
# ======================================================

# Lecture des datasets (sep=";" car les fichiers utilisent le point-virgule comme séparateur)

x_train = pd.read_csv("x_train.csv", sep=";")
y_train = pd.read_csv("y_train.csv", sep=";")
x_test  = pd.read_csv("x_test.csv", sep=";")

# Fusion des variables explicatives et de la cible

df = x_train.merge(y_train, on="ID")

# Sélection des colonnes contenant les rendements

returns_cols = [c for c in x_train.columns if c.startswith("r")]

In [3]:
# ======================================================
# FEATURE ENGINEERING
# ======================================================

# Construction de variables explicatives à partir des 53 rendements intraday
# On crée plusieurs types de features :
# - statistiques globales
# - momentum court terme
# - structure intraday
# - volatilité
# - pression acheteurs/vendeurs
# - tendances (trend)
# - interactions entre variables


def build_features(df, returns_cols):

    X = pd.DataFrame(index=df.index)

    R = df[returns_cols]


    # ===== STATISTIQUES GLOBALES =====
    # Capture la distribution globale des rendements

    X["mean"] = R.mean(axis=1)
    X["std"] = R.std(axis=1)
    X["min"] = R.min(axis=1)
    X["max"] = R.max(axis=1)

    X["range"] = X["max"] - X["min"]

    X["sum"] = R.sum(axis=1)


    # ===== MOMENTUM FINAL =====
    # Capture la dynamique récente du prix

    X["last_return"] = R.iloc[:, -1]

    X["last3_mean"] = R.iloc[:, -3:].mean(axis=1)
    X["last5_mean"] = R.iloc[:, -5:].mean(axis=1)

    X["momentum_short"] = R.iloc[:, -1] - R.iloc[:, -5]

    X["momentum_last10"] = R.iloc[:, -10:].mean(axis=1)


    # ===== STRUCTURE INTRADAY =====
    # Compare les rendements du début et de la fin de séance

    X["morning_mean"] = R.iloc[:, :10].mean(axis=1)

    X["midday_mean"] = R.iloc[:, 10:30].mean(axis=1)

    X["preclose_mean"] = R.iloc[:, -10:].mean(axis=1)

    X["intraday_shift"] = X["preclose_mean"] - X["morning_mean"]


    # ===== VOLATILITÉ =====
    # Mesure la dispersion des rendements

    X["vol_total"] = R.std(axis=1)

    X["vol_last10"] = R.iloc[:, -10:].std(axis=1)

    X["vol_first10"] = R.iloc[:, :10].std(axis=1)

    X["vol_ratio"] = X["vol_last10"] / (X["vol_first10"] + 1e-6)


    # ===== PRESSION ACHETEUR / VENDEUR =====
    # Mesure l'équilibre entre les mouvements positifs et négatifs des rendements intraday

    # Proportion de rendements positifs
    X["pos_ratio"] = (R > 0).mean(axis=1)

    # Proportion de rendements négatifs
    X["neg_ratio"] = (R < 0).mean(axis=1)

    # Pression acheteuse récente (fin de séance)
    X["pos_last10"] = (R.iloc[:, -10:] > 0).mean(axis=1)


    # ===== TREND =====
    # Estime la pente des rendements via une régression linéaire

    t = np.arange(R.shape[1])

    X["trend"] = R.apply(lambda x: np.polyfit(t, x, 1)[0], axis=1)

    X["trend_last10"] = R.iloc[:, -10:].apply(
        lambda x: np.polyfit(np.arange(10), x, 1)[0],
        axis=1
    )

    X["trend_acceleration"] = X["trend_last10"] - X["trend"]


    # ===== INTERACTIONS =====
    # Combinaison de plusieurs variables pour capturer des effets non linéaires

    X["momentum_vol"] = X["momentum_short"] * X["vol_total"]

    X["trend_vol"] = X["trend_last10"] * X["vol_last10"]

    X["range_vol"] = X["range"] * X["vol_total"]


    # Nettoyage

    X = X.replace([np.inf, -np.inf], np.nan)

    X = X.fillna(0)

    return X


# Les variables construites permettent de capturer différentes dimensions du comportement des rendements intraday :

#  - le niveau moyen des rendements
#  - le momentum de court terme
#  - la structure intraday de la séance
#  - la volatilité
#  - la pression acheteuse ou vendeuse
#  - les endances de marché

# Ces caractéristiques sont ensuite utilisées comme variables explicatives pour entraîner les modèles de classification.

In [4]:
# ======================================================
# CONSTRUCTION DES FEATURES
# ======================================================

# Application de la fonction de feature engineering sur les données d'entraînement.
# X_feat contient toutes les variables explicatives construites à partir des 53 rendements intraday
# Chaque ligne correspond à une observation (action × jour) et chaque colonne correspond à une feature calculée (statistiques, momentum, volatilité, trend..).

X_feat = build_features(df, returns_cols)

# Variable cible : direction du rendement de fin de journée
# reod ∈ {-1, 0, 1}

y = df["reod"]

# Construction des mêmes features pour le dataset de test (mêmes transformations que pour le train)

X_test_feat = build_features(x_test, returns_cols)

In [5]:
# ======================================================
# MODÈLES
# ======================================================

# LightGBM : modèle principal très performant pour les données tabulaires lgb = LGBMClassifier

lgb = LGBMClassifier(

    n_estimators=800,
    learning_rate=0.03,
    num_leaves=64,

    min_child_samples=50,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.5,
    reg_lambda=0.5,

    objective="multiclass",

    random_state=42,
    n_jobs=-1
)


# Random Forest : modèle complémentaire utilisé pour diversifier les prédictions et apporter de la robustesse au stacking

rf = RandomForestClassifier(

    n_estimators=100,

    max_depth=12,

    min_samples_leaf=40,

    random_state=42,

    n_jobs=-1
)


In [6]:
# ======================================================
# VALIDATION CROISÉE
# ======================================================
# Elle permet d'évaluer la robustesse du modèle en entrainant et testant le modèle sur différentes partitions du dataset.
# On utilise une StratifiedKFold afin de conserver la distribution des classes dans chaque fold : cela permet d'obtenir une estimation robuste de la performance du modèle.

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scores_lgb = []
scores_stack = []

print("Début CV")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_feat, y)):

    print("Fold", fold + 1)

    X_tr = X_feat.iloc[train_idx]
    X_val = X_feat.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]


    lgb.fit(X_tr, y_tr)
    rf.fit(X_tr, y_tr)


    proba_lgb = lgb.predict_proba(X_val)
    proba_rf  = rf.predict_proba(X_val)


    y_lgb = lgb.classes_[np.argmax(proba_lgb, axis=1)]

    scores_lgb.append(accuracy_score(y_val, y_lgb))


    proba_stack = 0.6 * proba_lgb + 0.4 * proba_rf

    y_stack = lgb.classes_[np.argmax(proba_stack, axis=1)]

    scores_stack.append(accuracy_score(y_val, y_stack))


print("CV LGBM =", np.mean(scores_lgb))
print("CV STACK =", np.mean(scores_stack))


Début CV
Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.280672 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6469
[LightGBM] [Info] Number of data points in the train set: 562199, number of used features: 28
[LightGBM] [Info] Start training from score -1.201407
[LightGBM] [Info] Start training from score -0.886658
[LightGBM] [Info] Start training from score -1.247580
Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.180951 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6470
[LightGBM] [Info] Number of data points in the train set: 562199, number of used features: 28
[LightGBM] [Info] Start training from score -1.201407
[LightGBM] [Info] Start training from score -0.886658
[LightGBM] [Info] Start training from score -1.247580
Fold 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the over

In [7]:
# ======================================================
# ENTRAINEMENT FINAL
# ======================================================
# Après la validation croisée on entraîne les modèles sur l'ensemble du dataset d'entraînement.

print("Entrainement final")

lgb.fit(X_feat, y)
rf.fit(X_feat, y)

Entrainement final
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.255889 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6470
[LightGBM] [Info] Number of data points in the train set: 843299, number of used features: 28
[LightGBM] [Info] Start training from score -1.201410
[LightGBM] [Info] Start training from score -0.886658
[LightGBM] [Info] Start training from score -1.247578


RandomForestClassifier(max_depth=12, min_samples_leaf=40, n_jobs=-1,
                       random_state=42)

In [8]:
# ======================================================
# PRÉDICTIONS
# ======================================================
# Les probabilités des deux modèles sont combinées via un stacking pondéré :
# 60% LightGBM
# 40% RandomForest
# Puis :
# Calibration légère des probabilités afin d'améliorer la prédiction directionnelle.

proba_lgb_test = lgb.predict_proba(X_test_feat)

proba_rf_test  = rf.predict_proba(X_test_feat)

# Le poids 0.6/0.4 a été choisi empiriquement après validation croisée
proba_stack_test = 0.6 * proba_lgb_test + 0.4 * proba_rf_test

proba_adj = proba_stack_test.copy()

# Léger boost pour la classe +1
proba_adj[:,2] *= 1.02

# Légère réduction pour -1
proba_adj[:,0] *= 0.99

# Renormalisation
proba_adj = proba_adj / proba_adj.sum(axis=1, keepdims=True)

y_pred_stack = lgb.classes_[np.argmax(proba_adj, axis=1)]

print(pd.Series(y_pred_stack).value_counts(normalize=True))

-1    0.364820
 1    0.343203
 0    0.291977
Name: proportion, dtype: float64


In [9]:
# ======================================================
# FICHIER DE SOUMISSION
# ======================================================
# Création du fichier final au format attendu par la plateforme de soumission : ID,reod.

submission = pd.DataFrame({
    "ID": x_test["ID"],
    "reod": y_pred_stack
})

submission.to_csv(
    "soumission.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

print("CSV sauvegardé")

CSV sauvegardé


In [10]:
# ======================================================
# VERIFICATION DU FORMAT ATTENDU
# ======================================================

with open("soumission.csv") as f:
    for _ in range(5):
        print(f.readline().strip())

ID,reod
1000000,-1
1000001,1
1000002,-1
1000003,1


**Résultat final**

Dans ce projet nous avons construit un pipeline de prédiction du rendement de fin de séance à partir des rendements intraday.

Les principales étapes ont été :

- construction de variables explicatives pertinentes
- entraînement de deux modèles de ML
- combinaison des prédictions via un stacking

Le modèle final atteint un score d'environ 0.4220, supérieur au benchmark fourni (0.4174), montrant ainsi que les features construites apportent une information prédictive utile.